In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path
from sklearn.metrics import mean_squared_error

In [2]:
data = np.load("../data/processed/windows_no_leak/window_60.npz")

X_train = data["X_train"]
y_train = data["y_train"]

X_test = data["X_test"]
y_test = data["y_test"]

print(X_train.shape)

(11841, 600)


In [3]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

In [4]:
class ANN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x)

In [5]:
input_dim = X_train.shape[1]

model = ANN(input_dim)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [6]:
epochs = 20
batch_size = 128

for epoch in range(epochs):

    perm = torch.randperm(X_train.size(0))

    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i+batch_size]

        xb = X_train[idx]
        yb = y_train[idx]

        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 1771.7784
Epoch 2, Loss: 1027.2964
Epoch 3, Loss: 738.9466
Epoch 4, Loss: 428.0605
Epoch 5, Loss: 320.8618
Epoch 6, Loss: 127.6256
Epoch 7, Loss: 254.7002
Epoch 8, Loss: 169.7174
Epoch 9, Loss: 125.6681
Epoch 10, Loss: 220.9910
Epoch 11, Loss: 89.5969
Epoch 12, Loss: 150.2721
Epoch 13, Loss: 154.9393
Epoch 14, Loss: 99.1054
Epoch 15, Loss: 115.2308
Epoch 16, Loss: 171.0562
Epoch 17, Loss: 99.6699
Epoch 18, Loss: 57.6641
Epoch 19, Loss: 86.5323
Epoch 20, Loss: 129.2768


In [7]:
model.eval()

with torch.no_grad():
    pred = model(X_test).numpy()

rmse = np.sqrt(mean_squared_error(y_test.numpy(), pred))

print("ANN RMSE:", rmse)

ANN RMSE: 14.997731863837622
